# Bước 6 · Nhận dạng người nói VIVOS và thử giọng nhân bản

**Không sinh lại âm thanh; không cần GPU.** Notebook dùng 4 CSV gốc Bước 3 và chạy toàn bộ bằng CPU.

Trên Kaggle: gắn một Dataset chứa `manifest_TA.csv`, `dac_trung_TA.csv`, `tham_chieu.csv` và `thiet_ke_cap.csv` vào **Input**. Tệp `DU_LIEU_DAU_VAO_BUOC6.zip` đi kèm có sẵn bốn CSV: giải nén trước rồi tải bốn CSV vào một Kaggle Dataset. Sau đó mở notebook, bật Internet nếu môi trường cần thư viện, chọn CPU và bấm **Run All**. Không sửa code.

Kết quả: `/kaggle/working/KET_QUA_BUOC6_NHAN_DANG.zip`, gồm báo cáo HTML, biểu đồ, split, CV, dự đoán và mô hình. Notebook kiểm tra trùng/lệch dữ liệu và dừng nếu phát hiện các tệp trùng tên nhưng khác nội dung.

**Giới hạn:** 25 người, 3 câu test/người, dữ liệu tái sử dụng từ T-A; không đại diện cho 65 người và không chứng minh đánh lừa xác thực thực tế.

In [ ]:
# BƯỚC 6: NHẬN DẠNG NGƯỜI NÓI THẬT VÀ THỬ CLONE – tự chạy, CPU
import os, json, hashlib, base64, io, zipfile, warnings, platform, sys
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
import sklearn, joblib

SEED=20261104
FILES=['manifest_TA.csv','dac_trung_TA.csv','tham_chieu.csv','thiet_ke_cap.csv']
# Kaggle: attach one Dataset containing all four CSVs, or the former Bước 3 output.
roots=[Path('/kaggle/input'), Path('/home/user/uploads'), Path.cwd()]
def find_file(name):
    hits=[]
    for root in roots:
        if root.exists():
            hits.extend(p for p in root.rglob(name) if p.is_file() and 'buoc6_ket_qua' not in str(p))
    # Deduplicate by resolved path, then check exact bytes: never silently select ambiguous input.
    hits=sorted({p.resolve() for p in hits},key=str)
    if not hits: raise FileNotFoundError('Thiếu '+name+'. Trên Kaggle: Add Input dataset chứa bốn CSV của Bước 3.')
    hashes={hashlib.sha256(p.read_bytes()).hexdigest() for p in hits}
    if len(hashes)>1: raise RuntimeError('Có nhiều bản '+name+' khác nội dung. Chỉ gắn một bộ Bước 3: '+str(hits))
    return hits[0]
paths={f:find_file(f) for f in FILES}
print('Tệp đầu vào:', *(str(x) for x in paths.values()),sep='\n  ')
base=Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/home/user/uploads')
out=base/'buoc6_ket_qua'; out.mkdir(parents=True,exist_ok=True)
manifest=pd.read_csv(paths[FILES[0]]); features=pd.read_csv(paths[FILES[1]]); refs=pd.read_csv(paths[FILES[2]]); design=pd.read_csv(paths[FILES[3]])
assert manifest.id_clip.is_unique and features.id_clip.is_unique, 'id_clip bị lặp'
assert set(manifest.id_clip)==set(features.id_clip), 'Manifest và đặc trưng không cùng id_clip'
d=manifest.merge(features,on='id_clip',validate='one_to_one')
assert set(d.nguon)=={'that','omnivoice','omdiom','vieneu'}
assert (d.groupby('pair_id').size()==4).all()
assert all(set(x)=={'that','omnivoice','omdiom','vieneu'} for _,x in d.groupby('pair_id').nguon)
assert (d.groupby('pair_id').speaker.nunique()==1).all() and (d.groupby('pair_id').text.nunique()==1).all()
assert refs.speaker.is_unique
assert not len(design.merge(refs[['speaker','ref_id']],left_on=['speaker','cau_id'],right_on=['speaker','ref_id'],how='inner')), 'Câu tham chiếu trùng câu phân loại'
real=d[d.nguon=='that'].copy().sort_values(['speaker','pair_id']).reset_index(drop=True)
assert real.speaker.nunique()==25 and len(real)==370 and d.shape[0]==1480, 'Số liệu không khớp bộ dữ liệu đã kiểm tra'
assert real.groupby('speaker').size().min()>=6
cols=[c for c in features.columns if c.startswith('F1_')]
assert len(cols)==52 and np.isfinite(d[cols].to_numpy(float)).all(), 'Thiếu/ lỗi đặc trưng MFCC'
# Group normalized identical text. Deterministically sample 3 per speaker. Search for a split
# that keeps every identical sentence wholly in one partition (including across speakers).
real['nhom_text']=real.text.fillna('').map(lambda x:' '.join(str(x).casefold().split()))
assert (real.nhom_text!='').all()
for offset in range(10000):
    rng=np.random.default_rng(SEED+offset); test_idx=[]
    for sp,g in real.groupby('speaker',sort=True): test_idx.extend(rng.choice(g.index,3,replace=False).tolist())
    test_set=set(test_idx)
    train_idx=[i for i in real.index if i not in test_set]
    if set(real.loc[train_idx,'nhom_text']).isdisjoint(real.loc[test_idx,'nhom_text']): break
else: raise RuntimeError('Không tạo được split bảo toàn nhóm câu trùng')
tr=real.loc[train_idx].copy().reset_index(drop=True); te=real.loc[sorted(test_idx)].copy().reset_index(drop=True)
assert len(tr)==295 and len(te)==75 and (te.groupby('speaker').size()==3).all()
# 3-fold CV, grouped by sentence. Ensure all 25 classes appear in every training and validation fold.
X=tr[cols].to_numpy(float); y=tr.speaker.to_numpy(); groups=tr.nhom_text.to_numpy(); classes=sorted(real.speaker.unique())
for off in range(1000):
    cv=StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=SEED+off)
    folds=list(cv.split(X,y,groups))
    if all(set(y[a])==set(classes) and set(y[b])==set(classes) for a,b in folds): break
else: raise RuntimeError('Không tìm được 3 fold đủ 25 lớp; cần chỉnh phương án CV, không đánh giá sai.')
assert all(set(groups[a]).isdisjoint(groups[b]) for a,b in folds)
models={
 'k-NN (k=3)': lambda: make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),KNeighborsClassifier(n_neighbors=3)),
 'SVM RBF (C=1)':lambda: make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),SVC(C=1,kernel='rbf')),
 'Gaussian NB':lambda: make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),GaussianNB()),
}
cvrows=[]
for name,ctor in models.items():
 for fold,(a,b) in enumerate(folds,1):
    model=ctor(); model.fit(X[a],y[a]); pred=model.predict(X[b]);
    cvrows.append({'mo_hinh':name,'fold':fold,'so_train':len(a),'so_val':len(b),'accuracy':accuracy_score(y[b],pred),'macro_f1':f1_score(y[b],pred,average='macro',zero_division=0)})
cvdf=pd.DataFrame(cvrows); summary=cvdf.groupby('mo_hinh',as_index=False).agg(cv_macro_f1=('macro_f1','mean'),cv_accuracy=('accuracy','mean'))
summary=summary.sort_values(['cv_macro_f1','mo_hinh'],ascending=[False,True]).reset_index(drop=True)
best=summary.iloc[0].mo_hinh
print('\nCV chỉ trên train:\n',summary.to_string(index=False),'\nMô hình chọn:',best)
# Chỉ sau khi cố định mô hình theo CV, xuất so sánh cả ba trên test thật.
real_test_rows=[]
for name,ctor in models.items():
    trial=ctor(); trial.fit(X,y)
    yp=trial.predict(te[cols].to_numpy(float)); yt=te.speaker.to_numpy()
    real_test_rows.append({'mo_hinh':name,'chon_theo_CV':name==best,'so_clip':len(te),'accuracy':accuracy_score(yt,yp),'precision_macro':precision_score(yt,yp,labels=classes,average='macro',zero_division=0),'recall_macro':recall_score(yt,yp,labels=classes,average='macro',zero_division=0),'macro_f1':f1_score(yt,yp,labels=classes,average='macro',zero_division=0)})
real_test=pd.DataFrame(real_test_rows)
model=models[best](); model.fit(X,y)
test_pairs=set(te.pair_id); attack=d[d.pair_id.isin(test_pairs)].copy(); assert len(attack)==300
pred=model.predict(attack[cols].to_numpy(float)); attack['du_doan_speaker']=pred; attack['trung_ma_tham_chieu']=(attack.speaker==pred)
rows=[]
for src in ['that','omnivoice','omdiom','vieneu']:
 x=attack[attack.nguon==src]; truth=x.speaker; p=x.du_doan_speaker
 rows.append({'nguon':src,'so_clip':len(x),'trung_ma':int((truth==p).sum()),'ty_le_trung_ma':accuracy_score(truth,p),'precision_macro':precision_score(truth,p,labels=classes,average='macro',zero_division=0),'recall_macro':recall_score(truth,p,labels=classes,average='macro',zero_division=0),'macro_f1':f1_score(truth,p,labels=classes,average='macro',zero_division=0)})
result=pd.DataFrame(rows)
print('\nKết quả test của mô hình đã chọn:\n',result.to_string(index=False))
# Log split and per clip, avoid retaining 75 feature columns in prediction export.
real['chia_buoc6']=np.where(real.pair_id.isin(test_pairs),'test','train')
foldmap={int(i):fold for fold,(a,b) in enumerate(folds,1) for i in b}
tr['fold_val_cv']=[foldmap[i] for i in range(len(tr))]
fold_by_pair=dict(zip(tr.pair_id,tr.fold_val_cv)); real['fold_val_cv']=real.pair_id.map(fold_by_pair).astype('Int64')
real[['pair_id','speaker','cau_id','text','nhom_text','chia_buoc6','fold_val_cv','tap']].rename(columns={'tap':'tap_TA_cu'}).to_csv(out/'chia_cau_buoc6.csv',index=False,encoding='utf-8-sig')
attack[['id_clip','pair_id','speaker','cau_id','text','nguon','tap','du_doan_speaker','trung_ma_tham_chieu']].rename(columns={'tap':'tap_TA_cu'}).to_csv(out/'du_doan_buoc6.csv',index=False,encoding='utf-8-sig')
cvdf.to_csv(out/'CV_train_theo_fold.csv',index=False,encoding='utf-8-sig');summary.to_csv(out/'chon_mo_hinh_train.csv',index=False,encoding='utf-8-sig');result.to_csv(out/'ket_qua_test.csv',index=False,encoding='utf-8-sig');real_test.to_csv(out/'so_sanh_ba_mo_hinh_test_that.csv',index=False,encoding='utf-8-sig')
per=attack.groupby(['nguon','speaker']).trung_ma_tham_chieu.agg(so_clip='size',trung_ma='sum',ty_le='mean').reset_index();per.to_csv(out/'ket_qua_tung_nguoi.csv',index=False,encoding='utf-8-sig')
joblib.dump({'model':model,'feature_names':cols,'speakers':classes,'mo_hinh':best,'warning':'closed-set 25 speaker; khong phai xac thuc danh tinh'},out/'nhan_dang_25_nguoi.joblib')
info={'muc_tieu':'Nhan dang 25 speaker VIVOS closed-set va do ti le gan dung ma clone','seed':SEED,'offset_chia':offset,'offset_cv':off,'n_train_that':len(tr),'n_test_that':len(te),'n_test_gia_moi_nguon':75,'model_chon_theo_cv':best,'feature':'52 MFCC + delta mean/std (F1)','tai_su_dung_du_lieu_TA':True,'nguoi_tham_chieu_khong_trung_cau_trong_thiet_ke':True,'file_sha256':{f:hashlib.sha256(p.read_bytes()).hexdigest() for f,p in paths.items()},'versions':{'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,'sklearn':sklearn.__version__},'han_che':['25 nguoi va 3 cau test moi nguoi, khong phai 65 nguoi','Dac trung va mot phan mau da duoc su dung trong T-A truoc, day la phan tich tiep theo khong phai tap holdout moi','OmDiOm fine-tune VIVOS, chua kiem tra chong lap tung ID voi du lieu TTS','Ty le gan dung ma la closed-set, khong phai ti le qua mat xac thuc thuc te','Cau OmniVoice co the bi doc sai/thieu; 5 cap tao loi truoc do bi loai dong bo']}
(out/'thong_tin_buoc6.json').write_text(json.dumps(info,ensure_ascii=False,indent=2),encoding='utf-8')
plt.rcParams.update({'font.family':'DejaVu Sans','axes.spines.top':False,'axes.spines.right':False,'figure.dpi':140})
fig,ax=plt.subplots(figsize=(8,4.2)); order=summary.sort_values('cv_macro_f1');bars=ax.barh(order.mo_hinh,order.cv_macro_f1*100,color=['#9cb4cf' if n!=best else '#2263b3' for n in order.mo_hinh]);ax.set(xlim=(0,100),xlabel='Macro-F1 trung bình trên CV train (%)',title='Chọn mô hình chỉ bằng dữ liệu train');ax.bar_label(bars,fmt='%.1f%%',padding=3);fig.tight_layout();fig.savefig(out/'01_chon_mo_hinh.png',bbox_inches='tight');plt.close(fig)
fig,ax=plt.subplots(figsize=(8,4.2)); names=['Thật','OmniVoice','OmDiOm','VieNeu']; vals=result.ty_le_trung_ma.to_numpy()*100; bars=ax.bar(names,vals,color=['#1f8b67','#ed9b40','#7753b6','#2184bf']);ax.set(ylim=(0,110),ylabel='Tỷ lệ gán đúng mã người (%)',title='Bộ nhận dạng đã chọn: cùng 75 câu test');ax.bar_label(bars,labels=[f'{int(r.trung_ma)}/{r.so_clip}\n{r.ty_le_trung_ma*100:.1f}%' for r in result.itertuples()],padding=3);fig.tight_layout();fig.savefig(out/'02_thu_clone.png',bbox_inches='tight');plt.close(fig)
x=attack[attack.nguon=='that']; cm=confusion_matrix(x.speaker,x.du_doan_speaker,labels=classes);fig,ax=plt.subplots(figsize=(11,9));im=ax.imshow(cm,cmap='Blues');ax.set_xticks(range(len(classes)),[s.replace('VIVOSSPK','') for s in classes],rotation=90,fontsize=7);ax.set_yticks(range(len(classes)),[s.replace('VIVOSSPK','') for s in classes],fontsize=7);ax.set(xlabel='Người dự đoán (SPK)',ylabel='Người thật (SPK)',title='Ma trận nhầm lẫn · 75 clip thật test');fig.colorbar(im,ax=ax,label='Số clip');fig.tight_layout();fig.savefig(out/'03_ma_tran_that.png',bbox_inches='tight');plt.close(fig)
def embed(name):return 'data:image/png;base64,'+base64.b64encode((out/name).read_bytes()).decode()
def html_table(df):return df.to_html(index=False,border=0,float_format=lambda n:f'{n:.4f}',classes='data')
from html import escape
html=f'''<!doctype html><html lang="vi"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1"><title>Bước 6 · Nhận dạng người nói VIVOS</title><style>body{{font-family:system-ui,Arial,sans-serif;background:#f3f6fc;color:#17263d;margin:0;line-height:1.65}}main{{max-width:1100px;margin:auto;padding:32px}}header,.card{{background:white;border-radius:16px;padding:24px;margin:0 0 18px;box-shadow:0 4px 22px #172f5210}}header{{background:#15345d;color:white}}h1,h2{{line-height:1.25}}h2{{color:#184d86}}.warn{{border-left:5px solid #e8a64b;background:#fff8ec;padding:14px}}img{{max-width:100%;height:auto}}table{{border-collapse:collapse;width:100%;font-size:14px}}th,td{{padding:8px;border-bottom:1px solid #dce4ef;text-align:left}}th{{background:#edf3fb}}.scroll{{overflow:auto}}small{{color:#4b5f78}}code{{background:#eef2f8;padding:2px 4px}}</style></head><body><main><header><small style="color:#b9d7fd">THÍ NGHIỆM BỔ SUNG · DỮ LIỆU THỰC BƯỚC 3</small><h1>Bước 6: nhận dạng người nói và thử giọng nhân bản</h1><p>25 người VIVOS · 295 câu thật train · 75 câu thật test · ba bản giả trên cùng 75 câu</p></header>
<section class="card"><h2>1. Phạm vi và cách làm</h2><p>Dùng 52 đặc trưng MFCC và delta có sẵn trong <code>dac_trung_TA.csv</code>. Huấn luyện và chọn k-NN, SVM RBF hoặc Gaussian NB trên clip thật; chọn theo macro-F1 CV 3 fold chỉ trên train. Chuẩn hóa nằm trong pipeline của từng fold. Các câu trùng nội dung được giữ cùng nhóm khi chia train/test và CV.</p><p>Mô hình chọn: <b>{escape(best)}</b>. Mỗi người có 3 câu test; 75 bản giả từ mỗi bộ sinh là cùng cặp câu test. Tham chiếu không trùng ID câu trong thiết kế.</p><div class="warn"><b>Không phải thí nghiệm 65 người.</b> Dữ liệu được tái sử dụng từ T-A, một phần đã được xem ở thí nghiệm trước; kết quả là phân tích tiếp theo, không phải kiểm định hoàn toàn mới. Chưa đối chiếu được từng ID trong dữ liệu fine-tune OmDiOm.</div></section>
<section class="card"><h2>2. Chọn mô hình trên train</h2><div class="scroll">{html_table(summary)}</div><img src="{embed('01_chon_mo_hinh.png')}" alt="Macro F1 CV train"><small>Điểm trung bình 3 fold; CV theo nhóm nội dung câu, mỗi fold có đủ 25 người. Cố định mô hình và siêu tham số trước khi xem test.</small></section>
<section class="card"><h2>3. So sánh ba mô hình trên test thật</h2><div class="scroll">{html_table(real_test)}</div><p>Mô hình dùng cho thử clone được cố định bằng CV train trước khi xem bảng này; không chọn lại theo test. Bảng test chỉ là kết quả của cùng tập mẫu đã được tái sử dụng.</p></section>
<section class="card"><h2>4. Kết quả trên cùng câu test</h2><div class="scroll">{html_table(result)}</div><img src="{embed('02_thu_clone.png')}" alt="Tỷ lệ đúng mã theo nguồn"><p>Với bản thật, tỷ lệ là accuracy nhận dạng người. Với bản giả, đó là tỷ lệ được gán vào <b>mã người dùng làm tham chiếu</b>, không phải xác suất hoặc tỷ lệ vượt qua hệ thống xác thực thực tế. Phân loại đóng luôn trả về một trong 25 mã.</p><p>Văn bản một số clip OmniVoice đã được người dùng đánh giá là đọc sai hoặc thiếu; nội dung chưa được kiểm chứng cho toàn bộ test.</p></section>
<section class="card"><h2>5. Ma trận nhầm lẫn trên giọng thật</h2><img src="{embed('03_ma_tran_that.png')}" alt="Ma trận nhầm lẫn nhận dạng người"><small>Mỗi hàng là người thật, mỗi cột là mã dự đoán, nhãn rút gọn SPK. Mỗi người có 3 clip test nên từng hàng chỉ có ba quan sát.</small></section>
<section class="card"><h2>6. Giới hạn và tái lập</h2><ul><li>Chỉ 25/65 người theo đề xuất GitHub, 3 câu test/người; không suy rộng cho VIVOS hay giọng ngoài mẫu.</li><li>Cả tập train và test nhận dạng đều có cùng 25 người, khác với test 7 người chưa gặp của bộ phân loại thật/giả T-A. Không so sánh trực tiếp hai điểm.</li><li>OmDiOm fine-tune VIVOS, còn chưa kiểm chứng chồng lắp từng clip với tập huấn luyện TTS.</li><li>5 cặp sinh không đầy đủ đã bị loại đồng bộ trước đó; có nguy cơ sai lệch chọn mẫu. Trùng tần số lấy mẫu không xóa mọi dấu vết codec/kênh thu.</li><li>Cùng câu và người dự định nhân bản không bảo đảm tín hiệu giả thực sự giống người hoặc đọc đúng. Không dùng kết quả làm căn cứ xác thực danh tính.</li><li>Pipeline và phép thử dùng đặc trưng sẵn có; không cần tải/sinh lại audio và không có kiểm chứng nội dung bằng nghe mới ở Bước 6.</li></ul><p>Chi tiết mẫu, fold, dự đoán, phiên bản, SHA-256 và mô hình được xuất cùng báo cáo. Chỉ nạp tệp joblib đáng tin cậy.</p></section></main></body></html>'''
(out/'BAO_CAO_BUOC6.html').write_text(html,encoding='utf-8')
zip_path=base/'KET_QUA_BUOC6_NHAN_DANG.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
 for p in sorted(out.iterdir()):
  if p.is_file():z.write(p,arcname=p.name)
print('\nĐã lưu:',out,'\nZIP:',zip_path)
